# 🧺 한국 여행 장바구니 — Colab 버전

한국관광공사 국문 관광정보 API에서 시장·볼거리·축제를 추천하고, 선택한 장소를 가까운 순서의 여행 플랜으로 구성합니다.

위에서부터 셀을 차례대로 실행하세요. 마지막 셀에 나타나는 `public URL`을 누르면 앱이 열립니다.

## 1. 필요한 패키지 설치

In [ ]:
%pip install -q 'gradio>=6.0,<7' 'requests>=2.31,<3' 'python-dotenv>=1.0,<2'

## 2. 관광공사 인증키 입력

입력한 키는 화면에 표시되지 않으며 현재 Colab 런타임의 환경변수에만 보관됩니다. 이전에 대화에 공개한 키 대신 공공데이터포털에서 재발급한 키를 사용하세요.

In [ ]:
import os
from getpass import getpass

os.environ['TOUR_API_KEY'] = getpass('한국관광공사 API 인증키: ')
os.environ['KAKAO_REST_API_KEY'] = getpass('카카오 REST API 키: ')
print('인증키가 현재 런타임에 설정되었습니다.')

## 3. 관광공사 API 클라이언트

In [ ]:
from __future__ import annotations

import os
from datetime import date, timedelta
from typing import Any
from urllib.parse import unquote

import requests


BASE_URL = "https://apis.data.go.kr/B551011/KorService2"


class TourAPIError(RuntimeError):
    pass


class TourAPI:
    def __init__(self, service_key: str, timeout: int = 15):
        if not service_key:
            raise TourAPIError("`.env` 파일에 TOUR_API_KEY를 설정해 주세요.")
        # requests가 쿼리를 인코딩하므로 포털의 Encoding 키도 먼저 한 번 복원한다.
        self.service_key = unquote(service_key.strip())
        self.timeout = timeout

    @classmethod
    def from_env(cls) -> "TourAPI":
        return cls(os.getenv("TOUR_API_KEY", ""))

    def _get(self, endpoint: str, **params: Any) -> list[dict[str, Any]]:
        query = {
            "serviceKey": self.service_key,
            "MobileOS": "ETC",
            "MobileApp": "KoreaTripBasket",
            "_type": "json",
            "numOfRows": 30,
            "pageNo": 1,
            **params,
        }
        try:
            response = requests.get(f"{BASE_URL}/{endpoint}", params=query, timeout=self.timeout)
            response.raise_for_status()
            payload = response.json()
        except (requests.RequestException, ValueError) as exc:
            raise TourAPIError("네트워크 또는 응답 형식을 확인해 주세요.") from exc

        header = payload.get("response", {}).get("header", {})
        if str(header.get("resultCode", "")) not in {"0000", "0"}:
            message = header.get("resultMsg") or "알 수 없는 API 오류"
            raise TourAPIError(f"관광공사 API 오류: {message}")
        items = payload.get("response", {}).get("body", {}).get("items", {})
        if not items:
            return []
        result = items.get("item", [])
        return result if isinstance(result, list) else [result]

    def _areas(self, area_code: str | None = None) -> list[dict[str, Any]]:
        params = {"numOfRows": 100}
        if area_code:
            params["areaCode"] = area_code
        return self._get("areaCode2", **params)

    def resolve_area(self, query: str) -> tuple[str, str | None, str]:
        normalized = query.replace(" ", "").removesuffix("시").removesuffix("도")
        provinces = self._areas()
        for province in provinces:
            pname = province.get("name", "")
            pnorm = pname.replace(" ", "").removesuffix("특별시").removesuffix("광역시").removesuffix("특별자치도").removesuffix("도")
            if normalized in pname.replace(" ", "") or normalized == pnorm:
                return str(province["code"]), None, pname

        for province in provinces:
            for city in self._areas(str(province["code"])):
                cname = city.get("name", "")
                cnorm = cname.replace(" ", "").removesuffix("시").removesuffix("군").removesuffix("구")
                if normalized in cname.replace(" ", "") or normalized == cnorm:
                    return str(province["code"]), str(city["code"]), f"{province['name']} {cname}"
        raise TourAPIError(f"‘{query}’ 지역을 찾지 못했어요. 시·군·구 이름으로 다시 입력해 주세요.")

    @staticmethod
    def _clean(items: list[dict[str, Any]], category: str, limit: int) -> list[dict[str, Any]]:
        cleaned = []
        seen = set()
        for raw in items:
            content_id = str(raw.get("contentid", ""))
            title = str(raw.get("title", "")).strip()
            if not title or content_id in seen:
                continue
            seen.add(content_id)
            item = dict(raw)
            item["title"] = title
            item["category"] = category
            item["key"] = f"{category}:{content_id or title}"
            cleaned.append(item)
            if len(cleaned) >= limit:
                break
        return cleaned

    def recommend(self, destination: str, limit_each: int = 8) -> dict[str, Any]:
        area_code, sigungu_code, display_name = self.resolve_area(destination)
        common = {"areaCode": area_code, "arrange": "Q", "numOfRows": 50}
        if sigungu_code:
            common["sigunguCode"] = sigungu_code

        # 38=쇼핑. 전통시장 결과를 우선하고 부족하면 지역 쇼핑지를 보충한다.
        shopping = self._get("areaBasedList2", contentTypeId="38", **common)
        markets = [x for x in shopping if any(word in x.get("title", "") for word in ("시장", "장터", "마켓"))]
        markets.extend(x for x in shopping if x not in markets)

        # 12=관광지, 14=문화시설. 두 종류를 섞어 볼거리를 구성한다.
        sights = self._get("areaBasedList2", contentTypeId="12", **common)
        if len(sights) < limit_each:
            sights += self._get("areaBasedList2", contentTypeId="14", **common)

        start = date.today().strftime("%Y%m%d")
        end = (date.today() + timedelta(days=365)).strftime("%Y%m%d")
        festivals = self._get("searchFestival2", eventStartDate=start, eventEndDate=end, **common)

        items = (
            self._clean(markets, "market", limit_each)
            + self._clean(sights, "sight", limit_each)
            + self._clean(festivals, "festival", limit_each)
        )
        return {"display_name": display_name, "items": items}

    def nearby_places(
        self,
        center: dict[str, Any],
        content_type_id: str,
        limit: int = 2,
        radius: int = 3000,
    ) -> list[dict[str, Any]]:
        if not center.get("mapx") or not center.get("mapy"):
            return []
        items = self._get(
            "locationBasedList2",
            mapX=center["mapx"],
            mapY=center["mapy"],
            radius=radius,
            contentTypeId=content_type_id,
            arrange="E",
            numOfRows=max(limit, 5),
        )
        return [
            {
                "id": str(item.get("contentid", "")),
                "name": item.get("title", ""),
                "address": item.get("addr1", ""),
                "phone": item.get("tel", ""),
                "distance_m": int(float(item.get("dist") or 0)),
                "url": "",
            }
            for item in items[:limit]
            if item.get("title")
        ]

    def recommend_around_route(
        self,
        places: list[dict[str, Any]],
        day_end_keys: set[str] | None = None,
    ) -> dict[str, dict[str, Any]]:
        result: dict[str, dict[str, Any]] = {}
        for index, place in enumerate(places):
            entry: dict[str, Any] = {"restaurants": self.nearby_places(place, "39", limit=2)}
            if place["key"] in (day_end_keys or {places[-1]["key"]}):
                entry["stays"] = self.nearby_places(place, "32", limit=2, radius=5000)
            result[place["key"]] = entry
        return result


## 4. 카카오모빌리티 길찾기 API

In [ ]:
from __future__ import annotations

import os
from typing import Any

import requests


DIRECTIONS_URL = "https://apis-navi.kakaomobility.com/v1/directions"
LOCAL_CATEGORY_URL = "https://dapi.kakao.com/v2/local/search/category.json"


class KakaoAPIError(RuntimeError):
    pass


class KakaoMobility:
    def __init__(self, rest_api_key: str, timeout: int = 15):
        if not rest_api_key:
            raise KakaoAPIError("KAKAO_REST_API_KEY가 설정되지 않았습니다.")
        self.rest_api_key = rest_api_key.strip()
        self.timeout = timeout

    @classmethod
    def from_env(cls) -> "KakaoMobility":
        return cls(os.getenv("KAKAO_REST_API_KEY", ""))

    def route_leg(self, origin: dict[str, Any], destination: dict[str, Any]) -> dict[str, float]:
        params = {
            "origin": f"{origin['mapx']},{origin['mapy']}",
            "destination": f"{destination['mapx']},{destination['mapy']}",
            "priority": "RECOMMEND",
            "summary": "true",
        }
        try:
            response = requests.get(
                DIRECTIONS_URL,
                params=params,
                headers={"Authorization": f"KakaoAK {self.rest_api_key}"},
                timeout=self.timeout,
            )
            response.raise_for_status()
            payload = response.json()
        except (requests.RequestException, ValueError) as exc:
            raise KakaoAPIError("카카오 길찾기 요청에 실패했습니다.") from exc

        routes = payload.get("routes") or []
        if not routes or routes[0].get("result_code") not in (None, 0):
            message = routes[0].get("result_msg") if routes else payload.get("msg")
            raise KakaoAPIError(message or "카카오 길찾기 결과가 없습니다.")
        summary = routes[0].get("summary", {})
        return {
            "distance_km": float(summary.get("distance", 0)) / 1000,
            "duration_min": max(1, round(float(summary.get("duration", 0)) / 60)),
            "toll_fare": float(summary.get("fare", {}).get("toll", 0)),
        }

    def route_legs(self, places: list[dict[str, Any]]) -> list[dict[str, float] | None]:
        legs: list[dict[str, float] | None] = []
        for origin, destination in zip(places, places[1:]):
            if not all((origin.get("mapx"), origin.get("mapy"), destination.get("mapx"), destination.get("mapy"))):
                legs.append(None)
                continue
            try:
                legs.append(self.route_leg(origin, destination))
            except KakaoAPIError:
                legs.append(None)
        return legs

    def nearby_places(
        self,
        center: dict[str, Any],
        category_code: str,
        limit: int = 3,
        radius: int = 3000,
    ) -> list[dict[str, Any]]:
        if not center.get("mapx") or not center.get("mapy"):
            return []
        params = {
            "category_group_code": category_code,
            "x": center["mapx"],
            "y": center["mapy"],
            "radius": radius,
            "sort": "distance",
            "size": min(15, max(limit * 3, limit)),
        }
        try:
            response = requests.get(
                LOCAL_CATEGORY_URL,
                params=params,
                headers={"Authorization": f"KakaoAK {self.rest_api_key}"},
                timeout=self.timeout,
            )
            response.raise_for_status()
            documents = response.json().get("documents", [])
        except (requests.RequestException, ValueError) as exc:
            raise KakaoAPIError("카카오 주변 장소 검색에 실패했습니다.") from exc
        return [
            {
                "id": str(place.get("id", "")),
                "name": place.get("place_name", ""),
                "address": place.get("road_address_name") or place.get("address_name", ""),
                "phone": place.get("phone", ""),
                "distance_m": int(place.get("distance") or 0),
                "url": place.get("place_url", ""),
            }
            for place in documents[:limit]
            if place.get("place_name")
        ]

    def recommend_around_route(
        self,
        places: list[dict[str, Any]],
        day_end_keys: set[str] | None = None,
    ) -> dict[str, dict[str, Any]]:
        result: dict[str, dict[str, Any]] = {}
        used_restaurants: set[str] = set()
        used_stays: set[str] = set()
        for index, place in enumerate(places):
            restaurants = self.nearby_places(place, "FD6", limit=6)
            unique_restaurants = [x for x in restaurants if x["id"] not in used_restaurants][:2]
            used_restaurants.update(x["id"] for x in unique_restaurants)
            entry: dict[str, Any] = {"restaurants": unique_restaurants}

            is_day_end = place["key"] in (day_end_keys or {places[-1]["key"]})
            if is_day_end:
                stays = self.nearby_places(place, "AD5", limit=5, radius=5000)
                unique_stays = [x for x in stays if x["id"] not in used_stays][:2]
                used_stays.update(x["id"] for x in unique_stays)
                entry["stays"] = unique_stays
            result[place["key"]] = entry
        return result


## 5. Gradio 여행 챗봇

In [ ]:
from __future__ import annotations

import html
import math
import os
from datetime import date, timedelta
from typing import Any
from urllib.parse import quote

import gradio as gr
from dotenv import load_dotenv



load_dotenv()

CATEGORY_LABELS = {
    "market": "시장",
    "sight": "볼거리",
    "festival": "축제",
}
DAY_START = 9 * 60
DAY_END = 18 * 60
LUNCH_START = 12 * 60
DINNER_START = 18 * 60
MEAL_DURATION = 60


def new_state() -> dict[str, Any]:
    return {"destination": "", "recommendations": [], "basket": [], "route_customized": False}


def haversine(a: dict[str, Any], b: dict[str, Any]) -> float:
    """Return the great-circle distance in kilometres."""
    lat1, lon1 = math.radians(float(a["mapy"])), math.radians(float(a["mapx"]))
    lat2, lon2 = math.radians(float(b["mapy"])), math.radians(float(b["mapx"]))
    dlat, dlon = lat2 - lat1, lon2 - lon1
    value = math.sin(dlat / 2) ** 2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2) ** 2
    return 6371.0 * 2 * math.asin(math.sqrt(value))


def optimize_route(items: list[dict[str, Any]]) -> list[dict[str, Any]]:
    located = [x for x in items if x.get("mapx") and x.get("mapy")]
    unlocated = [x for x in items if not x.get("mapx") or not x.get("mapy")]
    if len(located) < 2:
        return located + unlocated

    center = {
        "mapx": sum(float(x["mapx"]) for x in located) / len(located),
        "mapy": sum(float(x["mapy"]) for x in located) / len(located),
    }
    current = min(located, key=lambda x: haversine(center, x))
    remaining = [x for x in located if x is not current]
    route = [current]
    while remaining:
        current = min(remaining, key=lambda x: haversine(route[-1], x))
        route.append(current)
        remaining.remove(current)
    return route + unlocated


def estimate_visit_minutes(item: dict[str, Any]) -> int:
    """Estimate a useful stay length from the Tourism API category and title."""
    title = item.get("title", "")
    if item.get("category") == "festival":
        return 180
    if any(word in title for word in ("테마파크", "놀이공원", "동물원", "아쿠아리움", "박물관", "미술관", "과학관")):
        return 150
    if any(word in title for word in ("둘레길", "트레킹", "올레", "자연휴양림", "수목원", "해수욕장")):
        return 180
    if any(word in title for word in ("공원", "마을", "거리", "섬")):
        return 120
    if item.get("category") == "market":
        return 90
    if any(word in title for word in ("사찰", "성당", "궁", "성", "전망대", "기념관")):
        return 75
    return 90


def clock(minutes: int) -> str:
    return f"{minutes // 60:02d}:{minutes % 60:02d}"


def build_schedule(
    route: list[dict[str, Any]],
    legs: list[dict[str, float] | None] | None = None,
) -> list[dict[str, Any]]:
    """Fit visits, travel and lunch into 09:00–18:00, then end with dinner and lodging."""
    if not route:
        return []
    events: list[dict[str, Any]] = []
    day, now = 1, DAY_START
    lunch_done = dinner_done = False

    def add_meal(name: str) -> None:
        nonlocal now, lunch_done, dinner_done
        start = max(now, LUNCH_START if name == "점심" else DINNER_START)
        events.append({"type": "meal", "day": day, "name": name, "start": start, "end": start + MEAL_DURATION})
        now = start + MEAL_DURATION
        if name == "점심":
            lunch_done = True
        else:
            dinner_done = True

    for index, item in enumerate(route):
        duration = estimate_visit_minutes(item)
        travel = 0
        travel_distance_km = None
        if index and legs and index - 1 < len(legs) and legs[index - 1]:
            travel = min(180, max(0, int(legs[index - 1]["duration_min"])))
            travel_distance_km = float(legs[index - 1]["distance_km"])
        elif index and item.get("mapx") and route[index - 1].get("mapx"):
            travel_distance_km = haversine(route[index - 1], item)
            travel = min(180, max(10, round(travel_distance_km / 35 * 60)))

        needed = travel + duration
        meal_buffer = (MEAL_DURATION if not lunch_done and now < DINNER_START else 0)
        if now + needed + meal_buffer > DAY_END and any(e.get("type") == "visit" and e["day"] == day for e in events):
            last_key = next(e["key"] for e in reversed(events) if e.get("type") == "visit" and e["day"] == day)
            if not dinner_done:
                add_meal("저녁")
            events.append({"type": "day_end", "day": day, "after_key": last_key})
            day += 1
            now, lunch_done, dinner_done = DAY_START, False, False

        if travel:
            events.append(
                {
                    "type": "travel",
                    "day": day,
                    "start": now,
                    "end": now + travel,
                    "minutes": travel,
                    "distance_km": travel_distance_km,
                    "is_driving": bool(index and legs and index - 1 < len(legs) and legs[index - 1]),
                }
            )
            now += travel
        if not lunch_done and (now >= LUNCH_START or now + duration > LUNCH_START):
            add_meal("점심")
        if not dinner_done and (now >= DINNER_START or now + duration > DINNER_START):
            add_meal("저녁")

        start = now
        end = min(DAY_END, start + duration)
        events.append(
            {
                "type": "visit",
                "day": day,
                "key": item["key"],
                "item": item,
                "start": start,
                "end": end,
                "duration": duration,
            }
        )
        now = end

    last_visit = next((e for e in reversed(events) if e["type"] == "visit"), None)
    if last_visit:
        if not dinner_done:
            add_meal("저녁")
        events.append({"type": "day_end", "day": day, "after_key": last_visit["key"]})
    return events


def schedule_day_end_keys(events: list[dict[str, Any]]) -> set[str]:
    return {e["after_key"] for e in events if e["type"] == "day_end" and e.get("after_key")}


def kakao_map_link(item: dict[str, Any]) -> str:
    if not item.get("mapx") or not item.get("mapy"):
        return ""
    name = quote(item["title"], safe="")
    return f"https://map.kakao.com/link/to/{name},{item['mapy']},{item['mapx']}"


def make_plan(
    items: list[dict[str, Any]],
    destination: str,
    legs: list[dict[str, float] | None] | None = None,
    nearby: dict[str, dict[str, Any]] | None = None,
    already_ordered: bool = False,
) -> str:
    if not items:
        return "먼저 추천 목록에서 여행지를 담아 주세요."
    route = items if already_ordered else optimize_route(items)
    lines = [
        f"## {destination} 여행 플랜",
        "예상 관람시간·이동시간을 합산해 관광은 09:00~18:00에 배치하고, 18시 이후에는 저녁과 숙박으로 마칩니다.\n",
    ]
    events = build_schedule(route, legs)
    current_day = 0
    previous_visit_index = -1
    for event in events:
        if event["day"] != current_day:
            current_day = event["day"]
            lines.append(f"### {current_day}일차")
        if event["type"] == "travel":
            mode = "자동차" if event.get("is_driving") else "예상 이동"
            distance = f"{event['distance_km']:.1f}km / " if event.get("distance_km") is not None else ""
            lines.append(
                f"- 🚗 **{clock(event['start'])}–{clock(event['end'])} 이동** · {mode} {distance}약 {event['minutes']}분"
            )
            continue
        if event["type"] == "meal":
            icon = "🍽️" if event["name"] == "점심" else "🌙"
            lines.append(f"- {icon} **{clock(event['start'])}–{clock(event['end'])} {event['name']} 식사 및 휴식**")
            continue
        if event["type"] == "day_end":
            local = (nearby or {}).get(event.get("after_key"), {})
            stays = local.get("stays", [])
            if stays:
                stay_parts = []
                for place in stays:
                    label = f"{place['name']} ({place['distance_m']}m)"
                    stay_parts.append(f"[{label}]({place['url']})" if place.get("url") else label)
                lines.append("- 🛏️ **오늘의 숙박:** " + " · ".join(stay_parts))
            continue
        item = event["item"]
        address = item.get("addr1") or "주소 정보 없음"
        map_link = kakao_map_link(item)
        link_text = f" · [카카오맵 길찾기]({map_link})" if map_link else ""
        lines.append(
            f"- 📍 **{clock(event['start'])}–{clock(event['end'])} {item['title']}** "
            f"({CATEGORY_LABELS[item['category']]}, 예상 {event['duration']}분)  \n"
            f"   {address}{link_text}"
        )
        local = (nearby or {}).get(item["key"], {})
        restaurants = local.get("restaurants", [])
        if restaurants:
            restaurant_parts = []
            for place in restaurants:
                label = f"{place['name']} ({place['distance_m']}m)"
                restaurant_parts.append(f"[{label}]({place['url']})" if place.get("url") else label)
            lines.append("  - 🍽️ **주변 식당:** " + " · ".join(restaurant_parts))
    if legs and any(legs):
        lines.append("\n> 자동차 거리와 시간은 카카오모빌리티 추천 경로 기준이며 교통 상황에 따라 달라질 수 있습니다.")
    else:
        lines.append("\n> 카카오 키가 없거나 길찾기에 실패해 직선거리를 표시했습니다. 이동시간은 카카오맵에서 확인하세요.")
    return "\n".join(lines)


def choice_label(item: dict[str, Any]) -> str:
    address = item.get("addr1") or "주소 없음"
    return f"[{CATEGORY_LABELS[item['category']]}] {item['title']} · {address}"


def cards_html(items: list[dict[str, Any]]) -> str:
    if not items:
        return "<p class='empty'>추천 결과가 없습니다.</p>"
    cards = []
    for item in items:
        image = item.get("firstimage") or item.get("firstimage2")
        image_tag = (
            f'<img src="{html.escape(image)}" alt="" loading="lazy">'
            if image
            else '<div class="no-image">사진 없음</div>'
        )
        cards.append(
            '<article class="place-card">'
            f"{image_tag}<div><span>{CATEGORY_LABELS[item['category']]}</span>"
            f"<h3>{html.escape(item['title'])}</h3>"
            f"<p>{html.escape(item.get('addr1') or '주소 정보 없음')}</p></div></article>"
        )
    return '<div class="card-grid">' + "".join(cards) + "</div>"


def gallery_items(items: list[dict[str, Any]]) -> list[tuple[str, str]]:
    """Build clickable gallery entries; only records with a photo are shown."""
    return [
        (
            item.get("firstimage") or item.get("firstimage2"),
            f"[{CATEGORY_LABELS[item['category']]}] {item['title']}\n{item.get('addr1') or '주소 정보 없음'}",
        )
        for item in items
        if item.get("firstimage") or item.get("firstimage2")
    ]


def basket_rows(state: dict[str, Any]) -> list[list[str]]:
    return [
        [CATEGORY_LABELS[x["category"]], x["title"], x.get("addr1") or ""]
        for x in state["basket"]
    ]


def initial_history() -> list[dict[str, str]]:
    return [{"role": "assistant", "content": "안녕하세요! 어디로 여행을 가시나요? 예: 부산, 전주시, 제주도"}]


def search_destination(message: str, history: list[dict[str, str]], state: dict[str, Any]):
    destination = message.strip()
    history = list(history or initial_history())
    if not destination:
        history.append({"role": "assistant", "content": "도시나 지역 이름을 입력해 주세요."})
        return history, state, gr.update(), [], ""

    history.append({"role": "user", "content": destination})
    try:
        api = TourAPI.from_env()
        result = api.recommend(destination)
    except TourAPIError as exc:
        history.append({"role": "assistant", "content": f"관광정보를 불러오지 못했어요. {exc}"})
        return history, state, gr.update(choices=[], value=[]), [], ""

    recommendations = result["items"]
    state = {
        "destination": result["display_name"],
        "recommendations": recommendations,
        "basket": [],
        "route_customized": False,
    }
    counts = {key: sum(x["category"] == key for x in recommendations) for key in CATEGORY_LABELS}
    history.append(
        {
            "role": "assistant",
            "content": (
                f"{result['display_name']}의 추천을 찾았어요: 시장 {counts['market']}곳, "
                f"볼거리 {counts['sight']}곳, 축제 {counts['festival']}곳입니다. "
                "아래에서 마음에 드는 곳을 체크하고 ‘여행 바구니에 담기’를 눌러 주세요."
            ),
        }
    )
    choices = [(choice_label(item), item["key"]) for item in recommendations]
    return history, state, gr.update(choices=choices, value=[]), gallery_items(recommendations), ""


def add_to_basket(keys: list[str], state: dict[str, Any], history: list[dict[str, str]]):
    selected = set(keys or [])
    existing = {x["key"] for x in state["basket"]}
    added = [x for x in state["recommendations"] if x["key"] in selected and x["key"] not in existing]
    state["basket"].extend(added)
    if added:
        state["route_customized"] = False
    history = list(history)
    if added:
        names = ", ".join(x["title"] for x in added)
        history.append({"role": "assistant", "content": f"여행 바구니에 담았어요: {names}"})
    else:
        history.append({"role": "assistant", "content": "새로 선택된 장소가 없어요."})
    return state, basket_rows(state), history, [x["key"] for x in state["basket"]]


def add_gallery_item(state: dict[str, Any], history: list[dict[str, str]], evt: gr.SelectData):
    pictured = [
        item
        for item in state["recommendations"]
        if item.get("firstimage") or item.get("firstimage2")
    ]
    if not isinstance(evt.index, int) or evt.index < 0 or evt.index >= len(pictured):
        return state, basket_rows(state), history, [x["key"] for x in state["basket"]]
    item = pictured[evt.index]
    existing = {x["key"] for x in state["basket"]}
    history = list(history)
    if item["key"] not in existing:
        state["basket"].append(item)
        state["route_customized"] = False
        history.append({"role": "assistant", "content": f"사진을 눌러 **{item['title']}**을(를) 여행 바구니에 담았어요."})
    else:
        history.append({"role": "assistant", "content": f"**{item['title']}**은(는) 이미 여행 바구니에 있어요."})
    return state, basket_rows(state), history, [x["key"] for x in state["basket"]]


def clear_basket(state: dict[str, Any]):
    state["basket"] = []
    state["route_customized"] = False
    return state, [], "", [], gr.update(choices=[], value=None)


def editor_choices(state: dict[str, Any]) -> list[tuple[str, str]]:
    return [(f"{index + 1}. {item['title']}", item["key"]) for index, item in enumerate(state["basket"])]


def reorder_basket(state: dict[str, Any], selected_key: str | None, delta: int) -> None:
    if not selected_key:
        return
    index = next((i for i, item in enumerate(state["basket"]) if item["key"] == selected_key), None)
    if index is None:
        return
    target = max(0, min(len(state["basket"]) - 1, index + delta))
    if target != index:
        state["basket"][index], state["basket"][target] = state["basket"][target], state["basket"][index]
    state["route_customized"] = True


def delete_from_basket(state: dict[str, Any], selected_key: str | None) -> None:
    if selected_key:
        state["basket"] = [item for item in state["basket"] if item["key"] != selected_key]
        state["route_customized"] = True


def compute_plan(state: dict[str, Any]) -> tuple[str, str]:
    if not state["basket"]:
        return "먼저 추천 목록에서 여행지를 담아 주세요.", ""
    route = list(state["basket"]) if state.get("route_customized") else optimize_route(state["basket"])
    state["basket"] = route
    state["route_customized"] = True


    legs = None
    nearby = None
    kakao_message = ""
    if len(route) > 1:
        try:
            kakao = KakaoMobility.from_env()
            legs = kakao.route_legs(route)
            day_end_keys = schedule_day_end_keys(build_schedule(route, legs))
            nearby = kakao.recommend_around_route(route, day_end_keys)
            if not any(legs):
                kakao_message = " 카카오 경로를 받지 못해 직선거리로 표시했어요."
        except KakaoAPIError:
            kakao_message = " 카카오 검색을 사용할 수 없어 관광공사 데이터로 주변 장소를 찾았어요."
            try:
                day_end_keys = schedule_day_end_keys(build_schedule(route, legs))
                nearby = TourAPI.from_env().recommend_around_route(route, day_end_keys)
            except TourAPIError:
                kakao_message = " 주변 식당·숙박 정보를 불러오지 못했어요."
    plan = make_plan(
        route,
        state.get("destination") or "선택 지역",
        legs=legs,
        nearby=nearby,
        already_ordered=True,
    )
    return plan, kakao_message


def build_plan(state: dict[str, Any], history: list[dict[str, str]]):
    plan, kakao_message = compute_plan(state)
    history = list(history)
    if state["basket"]:
        history.append({"role": "assistant", "content": f"담은 장소를 기준으로 여행 플랜을 만들었어요.{kakao_message}"})
    choices = editor_choices(state)
    selected = choices[0][1] if choices else None
    return plan, history, state, basket_rows(state), gr.update(choices=choices, value=selected)


def edit_plan(
    selected_key: str | None,
    state: dict[str, Any],
    history: list[dict[str, str]],
    action: str,
):
    if action == "up":
        reorder_basket(state, selected_key, -1)
    elif action == "down":
        reorder_basket(state, selected_key, 1)
    elif action == "delete":
        delete_from_basket(state, selected_key)
    plan, message = compute_plan(state)
    history = list(history)
    if action == "delete":
        history.append({"role": "assistant", "content": "선택한 장소를 일정에서 삭제하고 다시 계산했어요."})
    else:
        history.append({"role": "assistant", "content": "선택한 장소의 순서를 바꾸고 일정을 다시 계산했어요."})
    choices = editor_choices(state)
    next_value = selected_key if any(value == selected_key for _, value in choices) else (choices[0][1] if choices else None)
    return state, basket_rows(state), gr.update(choices=choices, value=next_value), plan, history


def move_plan_up(selected_key, state, history):
    return edit_plan(selected_key, state, history, "up")


def move_plan_down(selected_key, state, history):
    return edit_plan(selected_key, state, history, "down")


def delete_plan_item(selected_key, state, history):
    return edit_plan(selected_key, state, history, "delete")


CSS = """
.gradio-container {max-width: 1180px !important;}
.card-grid {display:grid; grid-template-columns:repeat(auto-fill,minmax(230px,1fr)); gap:14px; margin:10px 0 18px;}
.place-card {border:1px solid #e5e7eb; border-radius:16px; overflow:hidden; background:white; box-shadow:0 4px 16px #0000000b;}
.place-card img,.no-image {width:100%; height:145px; object-fit:cover; background:#eef2f0; display:flex; align-items:center; justify-content:center; color:#6b7280;}
.place-card div:last-child {padding:13px;}
.place-card span {font-size:12px; color:#087f5b; font-weight:700;}
.place-card h3 {font-size:17px; margin:5px 0;}
.place-card p {font-size:13px; color:#667085; margin:0;}
"""


def create_app() -> gr.Blocks:
    with gr.Blocks(title="한국 여행 장바구니") as demo:
        state = gr.State(new_state())
        gr.Markdown("# 🧺 한국 여행 장바구니\n관광공사 실시간 데이터에서 장소를 고르고, 가까운 동선으로 여행을 계획하세요.")
        chatbot = gr.Chatbot(value=initial_history(), height=360, label="여행 도우미")
        with gr.Row():
            message = gr.Textbox(placeholder="어디로 가시나요? 예: 강릉, 부산, 전주시", label="목적지", scale=5)
            send = gr.Button("추천 받기", variant="primary", scale=1)

        gr.Markdown("## 추천 장소\n사진을 클릭하면 즉시 여행 바구니에 담깁니다. 사진이 없는 장소는 아래 목록에서 선택하세요.")
        cards = gr.Gallery(
            label="사진을 눌러 바로 담기",
            columns=4,
            rows=2,
            height="auto",
            object_fit="cover",
            allow_preview=False,
            preview=False,
            buttons=[],
        )
        picks = gr.CheckboxGroup(label="사진이 없거나 여러 장소를 한꺼번에 담을 때 선택", choices=[])
        with gr.Row():
            add = gr.Button("🧺 여행 바구니에 담기", variant="primary")
            clear = gr.Button("바구니 비우기")
        basket = gr.Dataframe(headers=["종류", "장소", "주소"], datatype="str", interactive=False, label="여행 바구니")
        plan_button = gr.Button("✨ 여행 플랜 만들기", variant="primary")
        plan = gr.Markdown()
        gr.Markdown("## 일정 순서 편집\n장소를 선택한 다음 순서를 이동하거나 일정에서 삭제할 수 있습니다.")
        plan_item = gr.Dropdown(label="편집할 장소", choices=[], interactive=True)
        with gr.Row():
            move_up = gr.Button("⬆️ 앞으로 이동")
            move_down = gr.Button("⬇️ 뒤로 이동")
            delete_item = gr.Button("🗑️ 일정에서 삭제", variant="stop")

        inputs = [message, chatbot, state]
        outputs = [chatbot, state, picks, cards, message]
        send.click(search_destination, inputs, outputs)
        message.submit(search_destination, inputs, outputs)
        add.click(add_to_basket, [picks, state, chatbot], [state, basket, chatbot, picks])
        cards.select(add_gallery_item, [state, chatbot], [state, basket, chatbot, picks])
        clear.click(clear_basket, state, [state, basket, plan, picks, plan_item])
        plan_button.click(build_plan, [state, chatbot], [plan, chatbot, state, basket, plan_item])
        edit_inputs = [plan_item, state, chatbot]
        edit_outputs = [state, basket, plan_item, plan, chatbot]
        move_up.click(move_plan_up, edit_inputs, edit_outputs)
        move_down.click(move_plan_down, edit_inputs, edit_outputs)
        delete_item.click(delete_plan_item, edit_inputs, edit_outputs)
    return demo




## 6. 앱 실행

실행 결과의 `Running on public URL` 링크를 누르세요. Colab 셀을 중지하거나 런타임이 종료되면 링크도 종료됩니다.

In [ ]:
demo = create_app()
demo.launch(share=True, theme=gr.themes.Soft(), css=CSS, debug=True)